In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Protocol


# =========================================================
# Core Models
# =========================================================


@dataclass(frozen=True)
class FieldKeywordDefinition:
    keywords: tuple[str, ...] = field(default_factory=tuple)
    boost_score: float = 0.0


@dataclass(frozen=True)
class FieldDefinition:
    name: str
    description: str | None = None
    required: bool = False
    data_type: str = "string"
    children: tuple["FieldDefinition", ...] = field(default_factory=tuple)
    is_collection: bool = False
    metadata: dict[str, Any] = field(default_factory=dict)
    keyword_def: FieldKeywordDefinition = field(default_factory=FieldKeywordDefinition)


@dataclass(frozen=True)
class SegmentDefinition:
    name: str
    description: str | None = None
    fields: tuple[FieldDefinition, ...] = field(default_factory=tuple)
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass(frozen=True)
class DocumentSegmentDefinition:
    doc_type: str
    segments: tuple[SegmentDefinition, ...]
    aliases: tuple[str, ...] = field(default_factory=tuple)
    metadata: dict[str, Any] = field(default_factory=dict)


# =========================================================
# Source Interface
# =========================================================


class SegmentDefinitionSource(Protocol):
    def get(self, doc_type: str) -> DocumentSegmentDefinition: ...

    def exists(self, doc_type: str) -> bool: ...


# =========================================================
# In-memory Definitions
# =========================================================

QUOTATION_SEGMENT_DEFINITION = DocumentSegmentDefinition(
    doc_type="quotation",
    aliases=("steel_quotation", "quote"),
    segments=(
        SegmentDefinition(
            name="quotation_meta",
            description="견적서 상단 메타 정보 및 거래 당사자/거래 조건",
            metadata={
                "segment_keywords": ("견적서", "quotation", "quote"),
                "segment_keyword_boost_score": 0.12,
                "title_keywords": ("견적서", "quotation", "quote"),
                "title_boost_score": 0.22,
                "preferred_block_types": ("text", "group_kv"),
                "preferred_labels": ("section_header", "key_value_area", "text"),
            },
            fields=(
                FieldDefinition(
                    name="document_key",
                    description="문서 식별 키",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("견적번호", "no.", "문서번호", "document no"),
                        boost_score=0.08,
                    ),
                ),
                FieldDefinition(
                    name="supplier_name",
                    description="공급업체 상호",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("공급업체", "supplier", "판매처", "상호", "회사명"),
                        boost_score=0.05,
                    ),
                ),
                FieldDefinition(
                    name="supplier_contact_person",
                    description="공급업체 담당자명",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("담당자", "작성자", "contact person"),
                        boost_score=0.04,
                    ),
                ),
                FieldDefinition(
                    name="supplier_phone",
                    description="공급업체 연락처",
                    keyword_def=FieldKeywordDefinition(
                        keywords=(
                            "tel",
                            "전화",
                            "연락처",
                            "phone",
                            "fax",
                            "email",
                            "e-mail",
                        ),
                        boost_score=0.04,
                    ),
                ),
                FieldDefinition(
                    name="supplier_address",
                    description="공급업체 주소",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("주소", "address"),
                        boost_score=0.03,
                    ),
                ),
                FieldDefinition(
                    name="buyer_name",
                    description="고객사 상호",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("수신", "귀중", "貴中", "buyer", "고객사"),
                        boost_score=0.05,
                    ),
                ),
                FieldDefinition(
                    name="buyer_contact_person",
                    description="고객사 담당자명",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("수신", "참조", "담당자"),
                        boost_score=0.04,
                    ),
                ),
                FieldDefinition(
                    name="buyer_phone",
                    description="고객사 연락처",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("tel", "전화", "연락처"),
                        boost_score=0.03,
                    ),
                ),
                FieldDefinition(
                    name="buyer_address",
                    description="고객사 주소",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("주소", "address"),
                        boost_score=0.03,
                    ),
                ),
                FieldDefinition(
                    name="validity",
                    description="견적 유효기간",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("유효기간", "validity"),
                        boost_score=0.07,
                    ),
                ),
                FieldDefinition(
                    name="payment_terms",
                    description="대금지급 조건",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("지불조건", "결제조건", "payment"),
                        boost_score=0.07,
                    ),
                ),
                FieldDefinition(
                    name="delivery_terms",
                    description="운송 조건",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("인도조건", "운송조건", "delivery", "납기일자"),
                        boost_score=0.07,
                    ),
                ),
            ),
        ),
        SegmentDefinition(
            name="items",
            description="품목 테이블 및 합계",
            metadata={
                "segment_keywords": (
                    "품명",
                    "재질",
                    "규격",
                    "수량",
                    "단가",
                    "금액",
                    "합계",
                ),
                "segment_keyword_boost_score": 0.10,
                "preferred_block_types": ("table",),
                "preferred_labels": ("table",),
            },
            fields=(
                FieldDefinition(
                    name="rows",
                    description="품목 행 목록",
                    is_collection=True,
                    data_type="object",
                    children=(
                        FieldDefinition(
                            name="unit_price",
                            description="단가",
                            data_type="number",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("단가", "unit price", "price"),
                                boost_score=0.08,
                            ),
                        ),
                        FieldDefinition(
                            name="quantity",
                            description="수량",
                            data_type="number",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("수량", "qty", "q'ty", "quantity"),
                                boost_score=0.08,
                            ),
                        ),
                        FieldDefinition(
                            name="line_amount",
                            description="행 금액",
                            data_type="number",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("금액", "amount"),
                                boost_score=0.08,
                            ),
                        ),
                        FieldDefinition(
                            name="material_grade",
                            description="재질",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("재질", "grade", "material"),
                                boost_score=0.06,
                            ),
                        ),
                        FieldDefinition(
                            name="thickness",
                            description="두께",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("두께", "thickness"),
                                boost_score=0.05,
                            ),
                        ),
                        FieldDefinition(
                            name="size_text",
                            description="사이즈 원문",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("사이즈", "size", "길이", "폭", "length"),
                                boost_score=0.05,
                            ),
                        ),
                        FieldDefinition(
                            name="spec_text",
                            description="규격/스펙 원문",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("규격", "spec", "maker", "메이커"),
                                boost_score=0.06,
                            ),
                        ),
                        FieldDefinition(
                            name="vat",
                            description="부가세",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("부가세", "vat", "세액"),
                                boost_score=0.05,
                            ),
                        ),
                    ),
                ),
                FieldDefinition(
                    name="totals",
                    description="합계 정보",
                    data_type="object",
                    children=(
                        FieldDefinition(
                            name="total_amount",
                            description="총액",
                            data_type="number",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("합계", "총액", "부가세", "vat", "합계금액"),
                                boost_score=0.08,
                            ),
                        ),
                        FieldDefinition(
                            name="supply_amount",
                            description="공급가액",
                            data_type="number",
                            keyword_def=FieldKeywordDefinition(
                                keywords=(
                                    "공급가액",
                                    "공급가",
                                    "supply amount",
                                    "소계",
                                ),
                                boost_score=0.08,
                            ),
                        ),
                    ),
                ),
            ),
        ),
        SegmentDefinition(
            name="remarks",
            description="비고 및 자유 텍스트",
            metadata={
                "segment_keywords": (
                    "특기사항",
                    "비고",
                    "remarks",
                    "remark",
                    "note",
                    "특이사항",
                ),
                "segment_keyword_boost_score": 0.12,
                "title_keywords": (
                    "특기사항",
                    "비고",
                    "remarks",
                    "remark",
                    "note",
                    "특이사항",
                ),
                "title_boost_score": 0.20,
                "preferred_block_types": ("text", "group_kv"),
                "preferred_labels": ("section_header", "list", "text"),
            },
            fields=(
                FieldDefinition(
                    name="raw_text",
                    description="비고 원문",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("특기사항", "비고", "remark", "remarks", "note"),
                        boost_score=0.08,
                    ),
                ),
                FieldDefinition(
                    name="extracted_conditions",
                    description="추출된 조건 정보",
                    data_type="object",
                    required=False,
                    keyword_def=FieldKeywordDefinition(
                        keywords=("생산불가", "대체견적", "수급불가", "조건"),
                        boost_score=0.05,
                    ),
                ),
            ),
        ),
    ),
)

PURCHASE_ORDER_SEGMENT_DEFINITION = DocumentSegmentDefinition(
    doc_type="purchase_order",
    aliases=("po", "order_sheet"),
    segments=(
        SegmentDefinition(
            name="order_meta",
            description="발주서 상단 메타 정보",
            metadata={
                "segment_keywords": ("발주서", "purchase order", "po"),
                "segment_keyword_boost_score": 0.12,
                "title_keywords": ("발주서", "purchase order", "po"),
                "title_boost_score": 0.22,
                "preferred_block_types": ("text", "group_kv"),
                "preferred_labels": ("section_header", "key_value_area", "text"),
            },
            fields=(
                FieldDefinition(
                    name="order_no",
                    description="발주번호",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("발주번호", "order no", "po no"),
                        boost_score=0.08,
                    ),
                ),
                FieldDefinition(
                    name="supplier_name",
                    description="공급업체",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("공급업체", "supplier", "vendor"),
                        boost_score=0.05,
                    ),
                ),
                FieldDefinition(
                    name="buyer_name",
                    description="발주처",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("발주처", "buyer", "orderer"),
                        boost_score=0.05,
                    ),
                ),
            ),
        ),
        SegmentDefinition(
            name="items",
            description="발주 품목",
            metadata={
                "segment_keywords": ("품명", "수량", "단가", "금액"),
                "segment_keyword_boost_score": 0.10,
                "preferred_block_types": ("table",),
                "preferred_labels": ("table",),
            },
            fields=(
                FieldDefinition(
                    name="rows",
                    is_collection=True,
                    data_type="object",
                    children=(
                        FieldDefinition(
                            name="item_name",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("품명", "item"),
                                boost_score=0.06,
                            ),
                        ),
                        FieldDefinition(
                            name="quantity",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("수량", "qty", "quantity"),
                                boost_score=0.08,
                            ),
                        ),
                        FieldDefinition(
                            name="unit_price",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("단가", "unit price"),
                                boost_score=0.08,
                            ),
                        ),
                        FieldDefinition(
                            name="line_amount",
                            keyword_def=FieldKeywordDefinition(
                                keywords=("금액", "amount"),
                                boost_score=0.08,
                            ),
                        ),
                    ),
                ),
            ),
        ),
        SegmentDefinition(
            name="remarks",
            description="비고",
            metadata={
                "segment_keywords": ("비고", "remarks", "note"),
                "segment_keyword_boost_score": 0.10,
                "title_keywords": ("비고", "remarks", "note"),
                "title_boost_score": 0.18,
                "preferred_block_types": ("text", "group_kv"),
                "preferred_labels": ("section_header", "text", "list"),
            },
            fields=(
                FieldDefinition(
                    name="raw_text",
                    keyword_def=FieldKeywordDefinition(
                        keywords=("비고", "remarks", "note"),
                        boost_score=0.08,
                    ),
                ),
            ),
        ),
    ),
)


# =========================================================
# In-memory Source
# =========================================================


class InMemorySegmentDefinitionSource:
    """
    현재는 코드에 정의된 문서 타입 정의를 제공한다.
    나중에 DB 소스로 교체 가능하도록 source 인터페이스를 분리한다.
    """

    def __init__(self) -> None:
        definitions = (
            QUOTATION_SEGMENT_DEFINITION,
            PURCHASE_ORDER_SEGMENT_DEFINITION,
        )
        self._definitions_by_key: dict[str, DocumentSegmentDefinition] = {}

        for definition in definitions:
            self._register(definition)

    def _register(self, definition: DocumentSegmentDefinition) -> None:
        self._definitions_by_key[self._normalize(definition.doc_type)] = definition
        for alias in definition.aliases:
            self._definitions_by_key[self._normalize(alias)] = definition

    def get(self, doc_type: str) -> DocumentSegmentDefinition:
        normalized = self._normalize(doc_type)
        if normalized not in self._definitions_by_key:
            raise KeyError(f"Unsupported doc_type: {doc_type}")
        return self._definitions_by_key[normalized]

    def exists(self, doc_type: str) -> bool:
        return self._normalize(doc_type) in self._definitions_by_key

    @staticmethod
    def _normalize(doc_type: str) -> str:
        return doc_type.strip().lower()


# =========================================================
# Provider
# =========================================================


class SegmentDefinitionProvider:
    """
    service는 source가 파일/코드/DB 중 어디서 오든 모르고
    provider.get(doc_type)만 호출한다.
    """

    def __init__(
        self,
        source: SegmentDefinitionSource | None = None,
    ) -> None:
        self._source = source or InMemorySegmentDefinitionSource()

    def get(self, doc_type: str) -> DocumentSegmentDefinition:
        return self._source.get(doc_type)

    def exists(self, doc_type: str) -> bool:
        return self._source.exists(doc_type)


SegmentDefinitionProvider().get("quotation").segments[0].description


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Iterable, Protocol


@dataclass
class BlockContent:
    ref: str
    label: str | None
    content_type: str
    content: Any


@dataclass
class SegmentCandidate:
    segment_name: str
    score: float
    reasons: list[str] = field(default_factory=list)


@dataclass
class BlockSegmentRecommendation:
    block_ref: str
    label: str | None
    content_type: str
    primary_segment: str | None
    candidates: list[SegmentCandidate] = field(default_factory=list)


@dataclass
class SegmentBucket:
    segment_name: str
    block_refs: list[str] = field(default_factory=list)
    reasons: list[str] = field(default_factory=list)


@dataclass
class LayoutSegmentMappingRecommendation:
    doc_type: str
    allowed_segments: list[str]
    block_recommendations: list[BlockSegmentRecommendation] = field(
        default_factory=list
    )
    segment_buckets: list[SegmentBucket] = field(default_factory=list)
    unmapped_block_refs: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)


class BlockSegmentRule(Protocol):
    def supports(
        self,
        *,
        doc_type: str,
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> bool: ...

    def score(
        self,
        *,
        doc_type: str,
        doc_dict: dict[str, Any],
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> list[SegmentCandidate]: ...


class BaseBlockSegmentRule:
    def supports(
        self,
        *,
        doc_type: str,
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> bool:
        return True

    @staticmethod
    def _normalize_text(text: str) -> str:
        return " ".join(text.strip().lower().split())

    @classmethod
    def _text_of(cls, block: BlockContent) -> str:
        if block.content is None:
            return ""

        if isinstance(block.content, str):
            return block.content.strip()

        if isinstance(block.content, dict):
            parts: list[str] = []
            for k, v in block.content.items():
                key = "" if k is None else str(k).strip()
                val = "" if v is None else str(v).strip()
                if key:
                    parts.append(key)
                if val:
                    parts.append(val)
            return " ".join(parts).strip()

        if isinstance(block.content, list):
            parts: list[str] = []
            for row in block.content:
                if isinstance(row, list):
                    for cell in row:
                        cell_text = "" if cell is None else str(cell).strip()
                        if cell_text:
                            parts.append(cell_text)
                else:
                    row_text = "" if row is None else str(row).strip()
                    if row_text:
                        parts.append(row_text)
            return " ".join(parts).strip()

        return str(block.content).strip()

    @classmethod
    def _contains_keyword(cls, text: str, keyword: str) -> bool:
        normalized_text = cls._normalize_text(text)
        normalized_keyword = cls._normalize_text(keyword)
        return normalized_keyword in normalized_text

    @staticmethod
    def _candidate(
        segment_name: str,
        score: float,
        *reasons: str,
    ) -> SegmentCandidate:
        return SegmentCandidate(
            segment_name=segment_name,
            score=max(0.0, min(score, 0.99)),
            reasons=[reason for reason in reasons if reason],
        )


class PictureIgnoreRule(BaseBlockSegmentRule):
    def supports(
        self,
        *,
        doc_type: str,
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> bool:
        return block.content_type == "picture"

    def score(
        self,
        *,
        doc_type: str,
        doc_dict: dict[str, Any],
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> list[SegmentCandidate]:
        return []


class PageFooterIgnoreRule(BaseBlockSegmentRule):
    def supports(
        self,
        *,
        doc_type: str,
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> bool:
        return block.label == "page_footer"

    def score(
        self,
        *,
        doc_type: str,
        doc_dict: dict[str, Any],
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> list[SegmentCandidate]:
        return []


class DefinitionKeywordScoringRule(BaseBlockSegmentRule):
    def __init__(self, segment_keyword_provider) -> None:
        self._segment_keyword_provider = segment_keyword_provider

    def supports(
        self,
        *,
        doc_type: str,
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> bool:
        return block.content_type in {"text", "group_kv", "table"}

    def score(
        self,
        *,
        doc_type: str,
        doc_dict: dict[str, Any],
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
    ) -> list[SegmentCandidate]:
        text = self._text_of(block)
        if not text:
            return []

        segment_keyword_map = self._segment_keyword_provider(document_definition)
        candidates: list[SegmentCandidate] = []

        for segment in document_definition.segments:
            score = self._base_score_for_block(block=block, segment=segment)
            reasons: list[str] = []
            matched_keywords: list[str] = []

            keyword_bundle = segment_keyword_map.get(segment.name, {})

            for keyword, boost_score in keyword_bundle.get("field_keywords", []):
                if self._contains_keyword(text, keyword):
                    score += boost_score
                    matched_keywords.append(keyword)

            if matched_keywords:
                score += min(len(matched_keywords) * 0.015, 0.12)
                reasons.append(
                    f"field keywords matched: {', '.join(matched_keywords[:10])}"
                )

            segment_matches: list[str] = []
            for keyword, boost_score in keyword_bundle.get("segment_keywords", []):
                if self._contains_keyword(text, keyword):
                    score += boost_score
                    segment_matches.append(keyword)
            if segment_matches:
                reasons.append(
                    f"segment keywords matched: {', '.join(segment_matches[:8])}"
                )

            if block.label == "section_header":
                title_matches: list[str] = []
                for keyword, boost_score in keyword_bundle.get("title_keywords", []):
                    if self._contains_keyword(text, keyword):
                        score += boost_score
                        title_matches.append(keyword)
                if title_matches:
                    reasons.append(
                        f"title keywords matched: {', '.join(title_matches[:8])}"
                    )

            if not reasons:
                continue

            reasons.append(f"content_type={block.content_type}")
            if block.label:
                reasons.append(f"label={block.label}")

            candidates.append(self._candidate(segment.name, score, *reasons))

        return candidates

    @staticmethod
    def _base_score_for_block(
        *,
        block: BlockContent,
        segment: SegmentDefinition,
    ) -> float:
        score = 0.12

        preferred_block_types = tuple(segment.metadata.get("preferred_block_types", ()))
        preferred_labels = tuple(segment.metadata.get("preferred_labels", ()))

        if block.content_type in preferred_block_types:
            score += 0.18
        if block.label and block.label in preferred_labels:
            score += 0.08

        if block.content_type == "table" and segment.name == "items":
            score += 0.20
        if block.content_type == "group_kv" and segment.name.endswith("meta"):
            score += 0.12
        if block.label == "list" and segment.name == "remarks":
            score += 0.10

        return score


class SegmentRuleRegistry:
    def __init__(self, segment_keyword_provider) -> None:
        self._segment_keyword_provider = segment_keyword_provider

    def get_rules(self, doc_type: str) -> list[BlockSegmentRule]:
        return [
            PageFooterIgnoreRule(),
            PictureIgnoreRule(),
            DefinitionKeywordScoringRule(
                segment_keyword_provider=self._segment_keyword_provider,
            ),
        ]


class LayoutSegmentMappingService:
    def __init__(
        self,
        rule_registry: SegmentRuleRegistry | None = None,
        definition_provider: SegmentDefinitionProvider | None = None,
        primary_threshold: float = 0.50,
    ) -> None:
        self._definition_provider = definition_provider or SegmentDefinitionProvider()
        self._rule_registry = rule_registry or SegmentRuleRegistry(
            segment_keyword_provider=self._get_segment_keyword_bundle,
        )
        self._primary_threshold = primary_threshold

    def recommend(
        self,
        *,
        doc_type: str,
        doc_dict: dict[str, Any],
        blocks: list[BlockContent],
    ) -> LayoutSegmentMappingRecommendation:
        document_definition = self._definition_provider.get(doc_type)
        allowed_segments = tuple(
            segment.name for segment in document_definition.segments
        )
        normalized_blocks = self._normalize_blocks(blocks)
        rules = self._rule_registry.get_rules(doc_type)

        block_recommendations = [
            self._recommend_for_block(
                doc_type=doc_type,
                doc_dict=doc_dict,
                document_definition=document_definition,
                block=block,
                rules=rules,
            )
            for block in normalized_blocks
        ]

        segment_buckets = self._build_segment_buckets(
            allowed_segments=allowed_segments,
            block_recommendations=block_recommendations,
        )
        unmapped_block_refs = self._collect_unmapped_block_refs(block_recommendations)
        warnings = self._build_warnings(
            allowed_segments=allowed_segments,
            block_recommendations=block_recommendations,
            unmapped_block_refs=unmapped_block_refs,
        )

        return LayoutSegmentMappingRecommendation(
            doc_type=document_definition.doc_type,
            allowed_segments=list(allowed_segments),
            block_recommendations=block_recommendations,
            segment_buckets=segment_buckets,
            unmapped_block_refs=unmapped_block_refs,
            warnings=warnings,
        )

    def _get_segment_keyword_bundle(
        self,
        document_definition: DocumentSegmentDefinition,
    ) -> dict[str, dict[str, list[tuple[str, float]]]]:
        result: dict[str, dict[str, list[tuple[str, float]]]] = {}

        for segment in document_definition.segments:
            field_keywords: list[tuple[str, float]] = []
            segment_keywords: list[tuple[str, float]] = []
            title_keywords: list[tuple[str, float]] = []

            for field_def in self._walk_fields(segment.fields):
                for keyword in field_def.keyword_def.keywords:
                    if keyword:
                        field_keywords.append(
                            (
                                str(keyword).strip(),
                                float(field_def.keyword_def.boost_score),
                            )
                        )

            segment_boost = float(
                segment.metadata.get("segment_keyword_boost_score", 0.0) or 0.0
            )
            for keyword in segment.metadata.get("segment_keywords", ()):
                if keyword:
                    segment_keywords.append((str(keyword).strip(), segment_boost))

            title_boost = float(segment.metadata.get("title_boost_score", 0.0) or 0.0)
            for keyword in segment.metadata.get("title_keywords", ()):
                if keyword:
                    title_keywords.append((str(keyword).strip(), title_boost))

            result[segment.name] = {
                "field_keywords": self._dedupe_keyword_pairs(field_keywords),
                "segment_keywords": self._dedupe_keyword_pairs(segment_keywords),
                "title_keywords": self._dedupe_keyword_pairs(title_keywords),
            }

        return result

    def _walk_fields(
        self,
        fields: tuple[FieldDefinition, ...],
    ) -> Iterable[FieldDefinition]:
        for field_def in fields:
            yield field_def
            if field_def.children:
                yield from self._walk_fields(field_def.children)

    @staticmethod
    def _dedupe_keyword_pairs(
        pairs: list[tuple[str, float]],
    ) -> list[tuple[str, float]]:
        seen: set[str] = set()
        result: list[tuple[str, float]] = []

        for keyword, score in pairs:
            normalized_keyword = " ".join(keyword.strip().lower().split())
            if normalized_keyword in seen:
                continue
            seen.add(normalized_keyword)
            result.append((keyword, score))

        return result

    def _normalize_blocks(
        self,
        blocks: list[BlockContent],
    ) -> list[BlockContent]:
        normalized: list[BlockContent] = []

        for block in blocks:
            content = block.content

            if isinstance(content, str):
                content = content.strip()

            elif isinstance(content, dict):
                normalized_dict: dict[str, Any] = {}
                for k, v in content.items():
                    nk = "" if k is None else str(k).strip()
                    nv = "" if v is None else str(v).strip()
                    normalized_dict[nk] = nv
                content = normalized_dict

            elif isinstance(content, list):
                normalized_list: list[Any] = []
                for row in content:
                    if isinstance(row, list):
                        normalized_row = [
                            "" if cell is None else str(cell).strip() for cell in row
                        ]
                        normalized_list.append(normalized_row)
                    else:
                        normalized_list.append("" if row is None else str(row).strip())
                content = normalized_list

            normalized.append(
                BlockContent(
                    ref=block.ref.strip(),
                    label=(
                        block.label.strip()
                        if isinstance(block.label, str)
                        else block.label
                    ),
                    content_type=block.content_type.strip(),
                    content=content,
                )
            )

        return normalized

    def _recommend_for_block(
        self,
        *,
        doc_type: str,
        doc_dict: dict[str, Any],
        document_definition: DocumentSegmentDefinition,
        block: BlockContent,
        rules: list[BlockSegmentRule],
    ) -> BlockSegmentRecommendation:
        raw_candidates: list[SegmentCandidate] = []

        for rule in rules:
            if not rule.supports(
                doc_type=doc_type,
                document_definition=document_definition,
                block=block,
            ):
                continue

            raw_candidates.extend(
                rule.score(
                    doc_type=doc_type,
                    doc_dict=doc_dict,
                    document_definition=document_definition,
                    block=block,
                )
            )

        allowed_segments = {segment.name for segment in document_definition.segments}
        raw_candidates = [
            candidate
            for candidate in raw_candidates
            if candidate.segment_name in allowed_segments
        ]

        merged_candidates = self._merge_candidates(raw_candidates)
        primary_segment = self._select_primary_segment(merged_candidates)

        return BlockSegmentRecommendation(
            block_ref=block.ref,
            label=block.label,
            content_type=block.content_type,
            primary_segment=primary_segment,
            candidates=merged_candidates,
        )

    def _merge_candidates(
        self,
        candidates: list[SegmentCandidate],
    ) -> list[SegmentCandidate]:
        merged: dict[str, SegmentCandidate] = {}

        for candidate in candidates:
            existing = merged.get(candidate.segment_name)
            if existing is None:
                merged[candidate.segment_name] = SegmentCandidate(
                    segment_name=candidate.segment_name,
                    score=candidate.score,
                    reasons=list(candidate.reasons),
                )
                continue

            existing.score = max(existing.score, candidate.score)
            existing.reasons.extend(candidate.reasons)

        result = list(merged.values())
        for item in result:
            item.reasons = self._dedupe_keep_order(item.reasons)

        result.sort(key=lambda x: x.score, reverse=True)
        return result

    def _select_primary_segment(
        self,
        candidates: list[SegmentCandidate],
    ) -> str | None:
        if not candidates:
            return None

        best = candidates[0]
        if best.score < self._primary_threshold:
            return None

        return best.segment_name

    def _build_segment_buckets(
        self,
        *,
        allowed_segments: tuple[str, ...],
        block_recommendations: list[BlockSegmentRecommendation],
    ) -> list[SegmentBucket]:
        buckets: dict[str, SegmentBucket] = {
            segment_name: SegmentBucket(segment_name=segment_name)
            for segment_name in allowed_segments
        }

        for block_rec in block_recommendations:
            if block_rec.primary_segment is None:
                continue

            bucket = buckets[block_rec.primary_segment]
            bucket.block_refs.append(block_rec.block_ref)

            if block_rec.candidates:
                top_candidate = next(
                    (
                        candidate
                        for candidate in block_rec.candidates
                        if candidate.segment_name == block_rec.primary_segment
                    ),
                    block_rec.candidates[0],
                )
                bucket.reasons.extend(top_candidate.reasons)

        result: list[SegmentBucket] = []
        for segment_name in allowed_segments:
            bucket = buckets[segment_name]
            if not bucket.block_refs:
                continue
            bucket.reasons = self._dedupe_keep_order(bucket.reasons)
            result.append(bucket)

        return result

    def _collect_unmapped_block_refs(
        self,
        block_recommendations: list[BlockSegmentRecommendation],
    ) -> list[str]:
        return [
            block_rec.block_ref
            for block_rec in block_recommendations
            if block_rec.primary_segment is None
        ]

    def _build_warnings(
        self,
        *,
        allowed_segments: tuple[str, ...],
        block_recommendations: list[BlockSegmentRecommendation],
        unmapped_block_refs: list[str],
    ) -> list[str]:
        warnings: list[str] = []

        found_segments = {
            block_rec.primary_segment
            for block_rec in block_recommendations
            if block_rec.primary_segment is not None
        }

        for segment_name in allowed_segments:
            if segment_name not in found_segments:
                warnings.append(f"{segment_name} segment not found")

        if unmapped_block_refs:
            warnings.append(f"{len(unmapped_block_refs)} block(s) are unmapped")

        return warnings

    @staticmethod
    def _dedupe_keep_order(values: list[str]) -> list[str]:
        seen: set[str] = set()
        result: list[str] = []
        for value in values:
            if value in seen:
                continue
            seen.add(value)
            result.append(value)
        return result

In [ ]:
# provider = SegmentDefinitionProvider(source=DbSegmentDefinitionSource())
service = LayoutSegmentMappingService()
result = service.recommend(
    doc_type="quotation",
    doc_dict={},
    blocks=blocks,
)
# print(f"type(result): {type(result)}")
print(f"doc_type: {result.doc_type}")
print(f"allowed_segments: {result.allowed_segments}")
for r in result.block_recommendations:
    print()
    print(f"block_ref: {r.block_ref}")
    print(f"label: {r.label}")
    print(f"content_type: {r.content_type}")
    print(f"primary_segment: {r.primary_segment}")
    for c in r.candidates:
        seg = getattr(c, "segment_type", None) or getattr(c, "segment_name", c)
        seg_str = seg.value if hasattr(seg, "value") else seg
        print(f"  candidate segment_type: {seg_str}")
        print(f"  candidate score: {c.score}")
        print(f"  candidate reasons: {c.reasons}")

In [ ]:
# 세그먼트별로 출력 (segment_buckets 기준) + block별 실제 텍스트
def _seg_str(x):
    if x is None:
        return None
    return x.value if hasattr(x, "value") else str(x)


def _print_block_content(blk):
    """content_type에 따라 블록 내용 출력: text는 그대로, group_kv는 dict 한 줄씩, table은 전부."""
    if blk is None or getattr(blk, "content", None) is None:
        print("      (내용 없음)")
        return
    ct = getattr(blk, "content_type", "")
    content = blk.content
    if ct == "text":
        print(f"      {content}")
    elif ct == "group_kv" and isinstance(content, dict):
        for k, v in content.items():
            print(f"      {k}: {v}")
    elif ct == "table" and isinstance(content, list):
        for row in content:
            if isinstance(row, list):
                print("      ", row)
            else:
                print("      ", row)
    else:
        print(f"      {content}")


blocks_by_ref = {
    getattr(b, "ref", None): b for b in blocks if getattr(b, "ref", None) is not None
}

for bucket in result.segment_buckets:
    seg = getattr(bucket, "segment_type", None) or getattr(bucket, "segment_name", None)
    seg_str = _seg_str(seg)
    print(f"=== segment: {seg_str} ===")
    print(f"  block_refs: {bucket.block_refs}")
    if bucket.reasons:
        print(
            f"  reasons: {bucket.reasons[:3]}{'...' if len(bucket.reasons) > 3 else ''}"
        )
    for r in result.block_recommendations:
        r_primary = getattr(r, "primary_segment", None)
        if _seg_str(r_primary) == seg_str:
            print(
                f"    - {r.block_ref} | label={r.label} | content_type={r.content_type}"
            )
            _print_block_content(blocks_by_ref.get(r.block_ref))
    print()

if result.unmapped_block_refs:
    print("=== unmapped (미매핑) ===")
    print(f"  block_refs: {result.unmapped_block_refs}")
    for ref in result.unmapped_block_refs:
        blk = blocks_by_ref.get(ref)
        if blk:
            print(f"    - {ref}:")
            _print_block_content(blk)